# Projeto C3 -  ANALISANDO DADOS DE PREÇOS DE CASAS NOS ESTADOS UNIDOS


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    silhouette_score
)

pd.set_option("display.max_columns", 100)

In [ ]:
caminho = Path("dados/train.csv")

if not caminho.exists():
    caminho = Path("train.csv")

df = pd.read_csv(caminho)

print("Quantidade de linhas:", df.shape[0])
print("Quantidade de colunas:", df.shape[1])

df.head()

## 1. Análise exploratória de dados

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
numericas = df.select_dtypes(include=["int64", "float64"]).columns
categoricas = df.select_dtypes(include=["object"]).columns

print("Colunas numéricas:", len(numericas))
print("Colunas categóricas:", len(categoricas))

In [ ]:
faltantes = df.isnull().sum().sort_values(ascending=False)
faltantes[faltantes > 0].head(20)

## 1.1 Análise da variável principal: preço de venda

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["SalePrice"], bins=30)
plt.title("Distribuição dos preços das casas")
plt.xlabel("Preço")
plt.ylabel("Quantidade")
plt.show()

print("Preço médio:", round(df["SalePrice"].mean(), 2))
print("Preço mediano:", round(df["SalePrice"].median(), 2))
print("Menor preço:", df["SalePrice"].min())
print("Maior preço:", df["SalePrice"].max())

## 1.2 Correlação entre variáveis

In [ ]:
correlacao = df[numericas].corr()["SalePrice"].sort_values(ascending=False)
correlacao.head(15)

In [ ]:
top_corr = correlacao.drop("SalePrice").head(10)

plt.figure(figsize=(10, 5))
plt.bar(top_corr.index, top_corr.values)
plt.title("Variáveis com maior correlação com o preço")
plt.xlabel("Variáveis")
plt.ylabel("Correlação")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 1.3 Verificação inicial de possíveis outliers

In [ ]:
colunas_boxplot = ["SalePrice", "GrLivArea", "GarageArea", "TotalBsmtSF", "OverallQual"]

for coluna in colunas_boxplot:
    if coluna in df.columns:
        plt.figure(figsize=(7, 3))
        plt.boxplot(df[coluna].dropna(), vert=False)
        plt.title("Boxplot - " + coluna)
        plt.xlabel(coluna)
        plt.show()

## 2. Feature Engineering

In [ ]:
dados = df.copy()

dados["TotalSF"] = (
    dados["TotalBsmtSF"].fillna(0) +
    dados["1stFlrSF"].fillna(0) +
    dados["2ndFlrSF"].fillna(0)
)

dados["TotalBath"] = (
    dados["FullBath"].fillna(0) +
    0.5 * dados["HalfBath"].fillna(0) +
    dados["BsmtFullBath"].fillna(0) +
    0.5 * dados["BsmtHalfBath"].fillna(0)
)

dados["HouseAge"] = dados["YrSold"] - dados["YearBuilt"]
dados["RemodAge"] = dados["YrSold"] - dados["YearRemodAdd"]

dados["HasGarage"] = np.where(dados["GarageArea"].fillna(0) > 0, 1, 0)
dados["HasBasement"] = np.where(dados["TotalBsmtSF"].fillna(0) > 0, 1, 0)

mediana = dados["SalePrice"].median()
dados["PriceCategory"] = np.where(dados["SalePrice"] >= mediana, "Alto", "Baixo")

dados[["SalePrice", "TotalSF", "TotalBath", "HouseAge", "RemodAge", "HasGarage", "HasBasement", "PriceCategory"]].head()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(dados["TotalSF"], dados["SalePrice"], alpha=0.5)
plt.title("Área total aproximada x preço")
plt.xlabel("Área total aproximada")
plt.ylabel("Preço")
plt.show()

## 2.1 Preparação dos dados para Machine Learning

In [ ]:
# Separando a variável alvo da regressão
y_reg = dados["SalePrice"]

# Removendo as colunas que não devem entrar como entrada
X = dados.drop(columns=["SalePrice", "PriceCategory"], errors="ignore")

# Separando numéricas e categóricas
colunas_num = X.select_dtypes(include=["int64", "float64"]).columns
colunas_cat = X.select_dtypes(include=["object"]).columns

# Preenchendo valores faltantes
X[colunas_num] = X[colunas_num].fillna(X[colunas_num].median())

for coluna in colunas_cat:
    X[coluna] = X[coluna].fillna("Sem informação")

# Transformando texto em colunas numéricas
X_modelo = pd.get_dummies(X, drop_first=True)

print("Base pronta para modelos:", X_modelo.shape)
X_modelo.head()

## 3. Aprendizagem supervisionada: treinamento e avaliação de modelos de regressão

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_modelo,
    y_reg,
    test_size=0.2,
    random_state=42
)

modelos_reg = {
    "Regressão Linear": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42)
}

resultado_reg = []

for nome, modelo in modelos_reg.items():
    modelo.fit(X_train, y_train)
    previsao = modelo.predict(X_test)

    mae = mean_absolute_error(y_test, previsao)
    rmse = np.sqrt(mean_squared_error(y_test, previsao))
    r2 = r2_score(y_test, previsao)

    resultado_reg.append([nome, mae, rmse, r2])

resultado_reg = pd.DataFrame(resultado_reg, columns=["Modelo", "MAE", "RMSE", "R2"])
resultado_reg

In [ ]:
melhor_nome = resultado_reg.sort_values("R2", ascending=False).iloc[0]["Modelo"]
melhor_reg = modelos_reg[melhor_nome]

melhor_reg.fit(X_train, y_train)
previsao = melhor_reg.predict(X_test)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, previsao, alpha=0.5)
plt.title("Preço real x preço previsto")
plt.xlabel("Preço real")
plt.ylabel("Preço previsto")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])
plt.show()

print("Melhor modelo de regressão:", melhor_nome)

## 4. Aprendizagem supervisionada: treinamento e avaliação de modelos de classificação

In [ ]:
y_clf = dados["PriceCategory"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_modelo,
    y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

modelos_clf = {
    "Regressão Logística": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}

resultado_clf = []

for nome, modelo in modelos_clf.items():
    modelo.fit(X_train_c, y_train_c)
    previsao = modelo.predict(X_test_c)

    acc = accuracy_score(y_test_c, previsao)
    prec = precision_score(y_test_c, previsao, pos_label="Alto")
    rec = recall_score(y_test_c, previsao, pos_label="Alto")
    f1 = f1_score(y_test_c, previsao, pos_label="Alto")

    resultado_clf.append([nome, acc, prec, rec, f1])

resultado_clf = pd.DataFrame(resultado_clf, columns=["Modelo", "Acurácia", "Precisão", "Recall", "F1"])
resultado_clf

In [ ]:
melhor_nome_clf = resultado_clf.sort_values("F1", ascending=False).iloc[0]["Modelo"]
melhor_clf = modelos_clf[melhor_nome_clf]

melhor_clf.fit(X_train_c, y_train_c)
previsao_clf = melhor_clf.predict(X_test_c)

print("Melhor modelo de classificação:", melhor_nome_clf)
print()
print(classification_report(y_test_c, previsao_clf))

matriz = confusion_matrix(y_test_c, previsao_clf, labels=["Baixo", "Alto"])

plt.figure(figsize=(5, 4))
plt.imshow(matriz)
plt.title("Matriz de confusão")
plt.xticks([0, 1], ["Baixo", "Alto"])
plt.yticks([0, 1], ["Baixo", "Alto"])
plt.xlabel("Previsto")
plt.ylabel("Real")

for i in range(2):
    for j in range(2):
        plt.text(j, i, matriz[i, j], ha="center", va="center")

plt.colorbar()
plt.show()

## 5. Aprendizagem não supervisionada: aplicação de técnicas de clusterização

In [ ]:
colunas_cluster = [
    "OverallQual",
    "GrLivArea",
    "TotalSF",
    "TotalBath",
    "HouseAge",
    "GarageArea"
]

base_cluster = dados[colunas_cluster].copy()
base_cluster = base_cluster.fillna(base_cluster.median())

scaler = StandardScaler()
base_cluster_padrao = scaler.fit_transform(base_cluster)

inercia = []
silhueta = []

for k in range(2, 8):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    grupos = kmeans.fit_predict(base_cluster_padrao)

    inercia.append(kmeans.inertia_)
    silhueta.append(silhouette_score(base_cluster_padrao, grupos))

plt.figure(figsize=(7, 4))
plt.plot(range(2, 8), inercia, marker="o")
plt.title("Teste do número de clusters")
plt.xlabel("Quantidade de clusters")
plt.ylabel("Inércia")
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(range(2, 8), silhueta, marker="o")
plt.title("Silhueta")
plt.xlabel("Quantidade de clusters")
plt.ylabel("Valor da silhueta")
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
dados["Cluster"] = kmeans.fit_predict(base_cluster_padrao)

dados["Cluster"].value_counts()

In [ ]:
dados.groupby("Cluster")[colunas_cluster + ["SalePrice"]].mean().round(2)

## 6. Aprendizagem não supervisionada: aplicação de técnicas de redução de dimensionalidade

In [ ]:
pca = PCA(n_components=2)
componentes = pca.fit_transform(base_cluster_padrao)

dados["PCA1"] = componentes[:, 0]
dados["PCA2"] = componentes[:, 1]

plt.figure(figsize=(8, 5))
grafico = plt.scatter(dados["PCA1"], dados["PCA2"], c=dados["Cluster"], alpha=0.6)
plt.title("Clusters visualizados com PCA")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.colorbar(grafico, label="Cluster")
plt.show()

print("Variância explicada:", pca.explained_variance_ratio_)
print("Total explicado:", pca.explained_variance_ratio_.sum())

## 7. Aprendizagem não supervisionada: aplicação de técnicas de análise de associação

In [ ]:
base_assoc = dados.copy()

base_assoc["Qualidade"] = pd.cut(
    base_assoc["OverallQual"],
    bins=[0, 4, 6, 10],
    labels=["Baixa", "Média", "Alta"]
)

base_assoc["Area"] = pd.qcut(
    base_assoc["GrLivArea"],
    q=3,
    labels=["Pequena", "Média", "Grande"]
)

base_assoc["Idade"] = pd.cut(
    base_assoc["HouseAge"],
    bins=[-1, 20, 50, 200],
    labels=["Nova", "Intermediária", "Antiga"]
)

colunas_assoc = ["Neighborhood", "HouseStyle", "Qualidade", "Area", "Idade", "PriceCategory"]

base_assoc = base_assoc[colunas_assoc].astype(str)

transacoes = pd.get_dummies(base_assoc)

transacoes.head()

In [ ]:
try:
    from mlxtend.frequent_patterns import apriori, association_rules

    itens = apriori(transacoes, min_support=0.05, use_colnames=True)

    regras = association_rules(itens, metric="confidence", min_threshold=0.6)

    regras = regras.sort_values(["confidence", "lift"], ascending=False)

    regras[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)

except ImportError:
    print("A biblioteca mlxtend não está instalada.")
    print("Para instalar, use: pip install mlxtend")

## 8. Aprendizagem não supervisionada: aplicação de técnica de análise de outlier

In [ ]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.03)

resultado_lof = lof.fit_predict(base_cluster_padrao)

dados["Outlier"] = np.where(resultado_lof == -1, "Outlier", "Normal")

dados["Outlier"].value_counts()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(dados["GrLivArea"], dados["SalePrice"], alpha=0.5)
plt.title("Área habitável x preço")
plt.xlabel("Área habitável")
plt.ylabel("Preço")
plt.show()

dados[dados["Outlier"] == "Outlier"][
    ["SalePrice", "GrLivArea", "TotalSF", "OverallQual", "HouseAge", "GarageArea", "Outlier"]
].head(10)

## 9. Visualização de dados: apresentação dos resultados obtidos

In [ ]:
preco_qualidade = dados.groupby("OverallQual")["SalePrice"].mean()

plt.figure(figsize=(8, 5))
plt.bar(preco_qualidade.index, preco_qualidade.values)
plt.title("Preço médio por qualidade geral")
plt.xlabel("Qualidade geral")
plt.ylabel("Preço médio")
plt.show()

In [ ]:
preco_cluster = dados.groupby("Cluster")["SalePrice"].mean()

plt.figure(figsize=(7, 4))
plt.bar(preco_cluster.index.astype(str), preco_cluster.values)
plt.title("Preço médio por cluster")
plt.xlabel("Cluster")
plt.ylabel("Preço médio")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(resultado_reg["Modelo"], resultado_reg["R2"])
plt.title("Comparação dos modelos de regressão")
plt.xlabel("Modelo")
plt.ylabel("R²")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(resultado_clf["Modelo"], resultado_clf["F1"])
plt.title("Comparação dos modelos de classificação")
plt.xlabel("Modelo")
plt.ylabel("F1")
plt.show()